# Build a review policy from saved IFEval scores

Open this notebook from a CAAT checkout. Install the checkout with `python -m pip install ".[workflow]"` and launch Jupyter from the same environment so `caat` is on its PATH.

The bundled CSV has two confidence scores and two graders for each saved answer. This notebook uses the scores to choose a cutoff, checks it against both graders, and shows which answers enter the review queue.

In [ ]:
import csv
import json
import os
import subprocess
from pathlib import Path

from IPython.display import FileLink, Markdown, display

here = Path.cwd()
root = next(
    (folder for folder in (here, *here.parents) if (folder / "examples/ifeval.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook inside a CAAT checkout.")

config = root / "examples/ifeval.toml"
with (root / "examples/ifeval_graders.csv").open(newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))

print(f"{len(rows)} saved answers")
for column in ("mean_token_logprob", "sequence_logprob", "strict_pass", "helm_strict_pass"):
    print(f"{column} = {rows[0][column]} on the first row")

## Choose a cutoff

The bundled configuration targets an accepted-set error rate of 40%. CAAT chooses a score on design rows, selects a cutoff on calibration rows, and applies that cutoff to held-out answers. `strict_pass` drives score and cutoff selection. `helm_strict_pass` checks the same accepted answers on the holdout. [IFEval configuration](ifeval.toml)

In a terminal, run `caat run examples/ifeval.toml` from the checkout root. The notebook uses the JSON output of the same command.

In [ ]:
completed = subprocess.run(
    ["caat", "run", str(config), "--json"],
    cwd=root,
    capture_output=True,
    encoding="utf-8",
)
if completed.returncode not in (0, 1):
    raise RuntimeError(completed.stderr or completed.stdout)

report = json.loads(completed.stdout)
policy = report["policy"]
print(f"Selected score = {report['selected_score']}")
print(f"Selected cutoff = {policy['selection']['selected_threshold']}")
print(f"Held-out policy confirmed = {policy['confirmed']}")

## Read the held-out check

Coverage is the share of held-out answers that the cutoff accepts. Error rate is the share of those accepted answers graded incorrect. At 95% confidence, each grader's one-sided upper bound must meet the configured target for the policy to be confirmed.

In [ ]:
def rate(value):
    return "unavailable" if value is None else f"{value:.1%}"

lines = [
    "| Grader | Accepted | Coverage | Errors | Error rate | Upper bound |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for grader, result in policy["heldout"]["graders"].items():
    lines.append(
        f"| {grader} | {result['accepted']}/{result['total']} | {rate(result['coverage'])} | "
        f"{result['errors']} | {rate(result['error_rate'])} | {rate(result['risk_upper_bound'])} |"
    )
display(Markdown("\n".join(lines)))

## Review the queued answers

The queue starts with the lowest-scored eligible answers. Open the HTML report for the complete queue and the comparison between graders.

In [ ]:
queue = report["review_queue"]
print(f"Queued {len(queue['entries'])} of {queue['eligible']} eligible answers")
for entry in queue["entries"][:5]:
    print(entry["id"])

html = Path(report["input"]["html"])
if not html.is_file():
    raise FileNotFoundError(html)
display(FileLink(os.path.relpath(html, here)))

## Run it on your data

Copy the TOML example and set `predictions`, `score_columns`, `correct_columns`, and `risk_target` for your dataset. Add `check_columns` for alternate graders. Resolve the CSV and HTML paths from the folder containing the TOML file. [Workflow settings](../docs/guide/workflow.md) · [CSV schema](../docs/guide/csv-format.md)